# Implementing `BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding` From Scratch

## Part 1: A Foundational Analysis of BERT

### Section 1: The Pre-BERT Landscape and the Need for Bidirectional Understanding

The central challenge in **Natural Language Processing (NLP)** has long been the development of models that can capture the rich, contextual meaning of human language. BERT was not an isolated invention but a direct and elegant solution to a well-defined set of limitations present in prior state-of-the-art models. <br>

**The Problem with Context-Free Embeddings:**<br>
Early successes in neural language modeling such as Word2Vec and GloVe revolutionized NLP by learning dense vector representations (embeddings) for words. These models operated on a simple principle: words that appear in similar contexts should have similar vector representations. This allowed models to capture semantic relationships, famously enabling vector arithmetic like:<br>
$king - man + woman ≈ queen.$<br>
But these embeddings were fundamentally static and context-free. The word "bank" for example, would have the exact same vector representation in "the river bank" and "the money in the bank." This inability to disambiguate words based on their surrounding texts a phenomenon known as **polysemy** posed a significant ceiling on the performance of downstream NLP tasks. The field needed representations that were dynamic and sensitive to context.<br>

Before BERT the dominant paradigm in NLP was to use language models that were unidirectional. Models like the canonical LSTMs or state-of-the-art GPT from OpenAI processed text sequentially either from left-to-right or right-to-left. <br>
Consider the sentence: *The man went to the bank to deposit money.* A left-to-right model would understand **bank** based only on *The man went to the...*. It couldn't use the crucial context **to deposit money** that comes after. This is a fundamental limitation because human language comprehension is not sequential it's holistic. We use the entire sentence to disambiguate meaning.

### Section-2: Assumptions, Goals, and Research Questions

- #### Primary Assumption
The paper assumes that a model pre-trained on a massive unlabeled text corpus can learn universal language representations that are beneficial for a wide variety of downstream NLP tasks (e.g., sentiment analysis, question answering). This builds upon prior work like Word2Vec and ELMo.

- #### Central Goal
To demonstrate that a deeply bidirectional model when pre-trained effectively will outperform unidirectional or shallowly bidirectional models across a wide range of NLP benchmarks and will establish a new state-of-the-art.

- #### Key Research Questions

1. Can the Transformer architecture originally designed for sequence-to-sequence tasks be adapted for language representation pre-training?
2. How can we train a truly bidirectional model given that seeing the whole sentence makes predicting the next word trivial?
3. Is learning inter-sentence relationships (like whether one sentence follows another) as important as learning intra-sentence relationships (word meanings)?
4. Can this pre-trained model be effectively fine-tuned for various tasks with minimal architectural changes, making it a general-purpose NLP backbone?

### 3. Mathematical Background

At its core `BERT` is simply the Encoder stack from the original Transformer paper ("Attention Is All You Need"). The key mathematical components are:

1. **Embeddings:** 

The input is represented as the sum of three embeddings:
- **Token Embeddings:** Standard embeddings that map each word (or sub-word token) in the vocabulary to a vector.
- **Segment Embeddings:** Vectors that indicate whether a token belongs to the first or second sentence in a pair $(E_A or E_B)$. This is crucial for the Next Sentence Prediction task.
- **Position Embeddings:** Since Transformers have no inherent sense of sequence these vectors are added to give the model information about the position of each token in the sequence. BERT uses learned positional embeddings.

## Part-2: Implementation from Scratch of BERT

### Section 1: Setup

In [ ]:
import math
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

### Section-2: Configuration